<a href="https://colab.research.google.com/github/IraSamsonova/nn/blob/main/AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()

        # Сверточные слои - признаки

        self.features = nn.Sequential(
            # Слой 1
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2), # Подвыборка для уменьшения размерности

            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2), # Финальная подвыборка перед полносвязными слоями
        )

        # Полносвязные слои - классификация

        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),

            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x) # Прямой проход через сверточные слои
        x = x.view(x.size(0), -1)
        x = self.classifier(x) # Классификация через полносвязные слои
        return x

In [ ]:
# Аугментация для обучающих данных
transform_train = transforms.Compose([
    transforms.Resize(256),  # Изменение размера
    transforms.CenterCrop(224),  # Центральное кадрирование
    transforms.RandomHorizontalFlip(),  # Случайное отражение по горизонтали
    transforms.ToTensor(),  # Конвертация в тензор
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Нормализация
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)  # Перемешивание для обучения
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)  # Без перемешивания для теста

In [ ]:
model = AlexNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()  # Функция потерь для классификации
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # Обнуление градиентов
        outputs = model(images)  # Прямой проход
        loss = criterion(outputs, labels)  # Вычисление потерь
        loss.backward()  # Обратное распространение
        optimizer.step()  # Обновление весов

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100 * correct / total


def test_epoch(model, loader):
    model.eval()  # Перевод модели в режим оценки
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100 * correct / total

In [ ]:
epochs = 10

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    test_loss, test_acc = test_epoch(model, test_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Test  Loss: {test_loss:.4f}, Test  Acc: {test_acc:.2f}%")

Epoch 1/10
Train Loss: 1.9521, Train Acc: 26.44%
Test  Loss: 1.4385, Test  Acc: 46.98%
Epoch 2/10
Train Loss: 1.3403, Train Acc: 51.32%
Test  Loss: 1.2211, Test  Acc: 57.38%
Epoch 3/10
Train Loss: 1.0379, Train Acc: 63.53%
Test  Loss: 0.9113, Test  Acc: 68.71%
Epoch 4/10
Train Loss: 0.8583, Train Acc: 70.02%
Test  Loss: 0.7746, Test  Acc: 73.39%
Epoch 5/10
Train Loss: 0.7382, Train Acc: 74.49%
Test  Loss: 0.7130, Test  Acc: 75.48%
Epoch 6/10
Train Loss: 0.6575, Train Acc: 77.19%
Test  Loss: 0.6326, Test  Acc: 78.59%
Epoch 7/10
Train Loss: 0.5822, Train Acc: 79.80%
Test  Loss: 0.5634, Test  Acc: 80.47%
Epoch 8/10
Train Loss: 0.5298, Train Acc: 81.56%
Test  Loss: 0.5865, Test  Acc: 80.14%
Epoch 9/10
Train Loss: 0.4851, Train Acc: 83.20%
Test  Loss: 0.5499, Test  Acc: 81.60%
Epoch 10/10
Train Loss: 0.4447, Train Acc: 84.64%
Test  Loss: 0.5181, Test  Acc: 82.31%
